# 02 · Extrusora: eficiencia energética por turno

## Enunciado

⚙️ *Contexto del Problema*: Una fábrica de juguetes de plástico requiere optimizar el proceso de extrusión para reducir el Consumo Específico de Energía ($kWh/ton$). El proceso presenta alta incertidumbre, ya que las curvas de temperatura de los calentadores (barril) son ajustadas manualmente por los operarios basándose en su experiencia.

La empresa requiere:
- Segmentar los turnos, cada turno se encuentra periodos de turnos de 12 horas (06:00 a 17:59 y 18:00 a 05:59). La empresa desea poder categorizar cada uno de los turnos para entender el comportamiento de la eficiencia energética (KPI) kwh/ton y para determinar si existe una diferencia estadísticamente significativa en el $KPI$ entre los diferentes turnos de de la Mañana y el Turno de la Noche, ¿Cuales son los turnos donde se presentaron mayores anomalias?.
- La empresa tiene la hipótesis  que existen curvas de temperatura en los calentadores que minimizan el consumo de energía en los motores, desarrolle una prueba para validar o rechazar esta hipótesis.




## Setup

In [7]:
%load_ext autoreload
%autoreload 2

import pandas as pd

pd.set_option("display.max_columns", 50)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from process.data import load_extrusora
from process.extrusora import *

df = load_extrusora()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44095 entries, 0 to 44094
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   fecha                  44095 non-null  datetime64[us]
 1   producto               43950 non-null  category      
 2   alimento_de_polvo_rot  43968 non-null  float64       
 3   temp_barril_2          43053 non-null  float64       
 4   temp_barril_3          43053 non-null  float64       
 5   temp_barril_4          43053 non-null  float64       
 6   temp_barril_5          43053 non-null  float64       
 7   temp_barril_6          43052 non-null  float64       
 8   temp_barril_7          43052 non-null  float64       
 9   temp_barril_8          43053 non-null  float64       
 10  temp_barril_9          43053 non-null  float64       
 11  temp_barril_10         43053 non-null  float64       
 12  temp_barril_11         43053 non-null  float64       
 13  melt_temp_su

## 1. Limpieza de Datos

Filtramos los momentos en que la extrusora no operaba y eliminamos outliers matemáticos (KPIs infinitos por divisiones entre cero).

In [9]:
df_cln = asignar_turno(df)
df_cln = limpiar_extrusora(df_cln)

print("Original:", len(df), "Limpio:", len(df_cln))
df_cln[["kpi", "alimento_de_polvo_rot", "md1_power"]].describe()

Original: 44095 Limpio: 35882


,kpi,alimento_de_polvo_rot,md1_power
count,35882.000000,35882.000000,35882.000000
mean,171.772363,28.435880,3579.596609
std,22.315783,2.550987,408.247564
min,93.308972,0.362000,26.390000
25%,159.321237,27.000000,3353.590000
50%,168.739665,28.016000,3587.850000
75%,180.069395,30.000000,3840.160000
max,252.444436,36.226000,4866.540000


## 2. Detección de Anomalías por Turno

Aplicamos Isolation Forest sobre las curvas de temperatura para detectar cuáles son los turnos que más se salen de control.

In [10]:
turnos_anomalos = ranking_turnos_anomalos(df_cln)
turnos_anomalos.head(10)

,turno_id,fecha_turno,turno,pct_anomalias,total_registros
1,2022-10-01_Mañana,2022-10-01,Mañana,1.000000,118
85,2022-11-13_Mañana,2022-11-13,Mañana,1.000000,27
62,2022-10-31_Noche,2022-10-31,Noche,0.995833,240
26,2022-10-13_Noche,2022-10-13,Noche,0.775000,240
27,2022-10-14_Mañana,2022-10-14,Mañana,0.708520,223
22,2022-10-11_Noche,2022-10-11,Noche,0.687500,240
61,2022-10-31_Mañana,2022-10-31,Mañana,0.616667,240
86,2022-11-13_Noche,2022-11-13,Noche,0.420833,240
170,2022-12-30_Mañana,2022-12-30,Mañana,0.387500,240
28,2022-10-14_Noche,2022-10-14,Noche,0.350000,240


## 3. Pruebas Estadísticas

### 3.1 Mañana vs Noche
Evaluamos si estadísticamente hay un turno más eficiente que el otro, incluso después de controlar por la mezcla de productos que procesa cada uno.

In [11]:
res_turnos = test_manana_vs_noche(df_cln)

for k, v in res_turnos.items():
    print(f"{k}: {v}")

mann_whitney_pvalue: 1.2078384825723325e-07
ols_pvalue_noche: 0.0002527873465167369
ols_coef_noche: -0.4406352222842432
media_manana: 172.18498335087492
media_noche: 171.36996319807497
mediana_manana: 169.05285980000002
mediana_noche: 168.49274915


La prueba confirma (p-value < 0.05) que el turno Noche es distinto. El coeficiente OLS muestra la magnitud del ahorro o sobrecosto del turno Noche frente al de la Mañana.

### 3.2 Curvas de Temperatura Óptimas
Agrupamos el perfil de calentamiento (Barril 2 al 11) mediante K-Means para probar la hipótesis de la gerencia: ¿Existen curvas de temperatura que minimizan el consumo de los motores?

In [12]:
res_temp = curvas_temperatura_optimas(df_cln, n_clusters=5)
print(f"P-Value Kruskal-Wallis: {res_temp['kruskal_pvalue']}")

res_temp["resumen_clusters"].sort_values("consumo_medio")

P-Value Kruskal-Wallis: 0.0


,cluster_temp,consumo_medio,kpi_medio,temp_barril_2_medio,temp_barril_11_medio,n
4,4,4042.744252,189.678229,170.130682,196.823392,286
3,3,4676.289710,163.307031,73.860757,192.504979,12968
0,0,4805.852840,168.464295,78.302083,193.729264,14222
2,2,5126.032344,176.264887,70.278523,222.574226,4052
1,1,5388.839163,202.434045,76.026778,200.350851,4354


¡La hipótesis es correcta! El P-value de 0.0 confirma la diferencia de consumos energéticos entre diferentes configuraciones de temperatura. El clúster que minimiza el consumo medio se puede usar como receta operativa recomendada.